In [1]:
import torch
print("PyTorch GPU available:", torch.cuda.is_available())
print("GPU device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch GPU available: True
GPU device name: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
# Imports
import os
from datasets import load_dataset, Audio, concatenate_datasets
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, TrainingArguments, Trainer
import torch
import torchaudio
from jiwer import wer
import torch
from dataclasses import dataclass
from typing import Dict, List, Union

In [ ]:
# Load Hindi + English datasets
common_voice_en = load_dataset("mozilla-foundation/common_voice_21_0", "en", split="train+validation")
common_voice_hi = load_dataset("mozilla-foundation/common_voice_22_0", "hi", split="train+validation")

print(f"English samples: {len(common_voice_en)}")
print(f"Hindi samples: {len(common_voice_hi)}")

In [ ]:
# Filter only clean samples with text
def filter_data(example):
    return example["sentence"] is not None and example["audio"] is not None

common_voice_en = common_voice_en.filter(filter_data)
common_voice_hi = common_voice_hi.filter(filter_data)

In [ ]:
# Combine datasets
dataset = concatenate_datasets([common_voice_en, common_voice_hi])
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
# Load Wav2Vec2 processor and model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-xlsr-53")
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-large-xlsr-53",
    vocab_size=len(processor.tokenizer),
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id
)

In [ ]:
# Preprocessing function
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

In [ ]:
# Data Collator
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features):
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt"
            )
        batch["labels"] = labels_batch["input_ids"]
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)

In [ ]:
# Metrics
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = torch.argmax(torch.tensor(pred_logits), dim=-1)
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer_score = wer(label_str, pred_str)
    print(f"Word Error Rate (WER): {wer_score:.4f}")
    return {"wer": wer_score}

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./asr_model",
    group_by_length=True,
    per_device_train_batch_size=8,
    evaluation_strategy="epoch",
    num_train_epochs=3,
    save_steps=400,
    eval_steps=400,
    logging_steps=100,
    learning_rate=1e-4,
    warmup_steps=500,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    push_to_hub=False
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=dataset.select(range(500)),
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics
)

# Train the model
print("Starting training...")
trainer.train()

In [ ]:
# Final Evaluation
print("Final Evaluation on 500 samples:")
trainer.evaluate()

In [ ]:
# Save model
os.makedirs("../models/", exist_ok=True)

model.save_pretrained("../models/")
processor.save_pretrained("../models/")
print("Model and processor saved to ../models/")